# 🌲 Sessão 07 — Hipsometria com Redes Neurais (MLP multi-feature)

> **Objetivo:** demonstrar empiricamente quando Redes Neurais superam significativamente modelos clássicos. Diferentemente da Sessão 06, aqui o baseline (Curtis) é **sub-especificado** — usa apenas DAP — enquanto o MLP recebe também idade e classe de sítio. A diferença de informação se traduz em ganho substancial.

---

## 📑 Sumário

1. [Por que esta sessão é diferente da 06?](#1-por-que-esta-sessão-é-diferente-da-06)
2. [Setup e Dados](#2-setup-e-dados)
3. [Baseline Clássico: Curtis (apenas DAP)](#3-baseline-clássico-curtis-apenas-dap)
4. [Engenharia de Features: One-Hot Encoding](#4-engenharia-de-features-one-hot-encoding)
5. [MLP Multi-feature](#5-mlp-multi-feature)
6. [Comparação Visual e Estatística](#6-comparação-visual-e-estatística)
7. [Análise por Classe de Sítio](#7-análise-por-classe-de-sítio)
8. [Síntese: Quando ML Vale a Pena?](#8-síntese-quando-ml-vale-a-pena)

## 1. Por que esta sessão é diferente da 06?

### O experimento na Sessão 06
- Baseline (Schumacher-Hall) usava DAP + H, com a forma funcional **correta** (gerou os dados).
- MLP usava as **mesmas** features.
- Resultado: baseline venceu (era estatisticamente ótimo).

### O experimento desta sessão
- Baseline (Curtis) usa **apenas DAP** — modelo intencionalmente sub-especificado.
- MLP usa **DAP + idade + classe de sítio**.
- Hipótese: a vantagem de informação se traduzirá em ganho mensurável.

### Por que Curtis com só DAP é o baseline canônico?

Na prática florestal, modelos hipsométricos por talhão são frequentemente univariados (Campos & Leite, 2017), porque idade e sítio são tratados em **camadas separadas** (ajustando uma equação por estrato). Comparar com um modelo único que recebe tudo é a forma honesta de demonstrar a **vantagem do ML**: ele integra naturalmente múltiplas fontes de variação.

## 2. Setup e Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from forestpy.data.loaders import load_pef_vinhedo
from forestpy.dendrometria.hipsometria import curtis
from forestpy.ml.preprocessing import StandardScalerForest
from forestpy.ml.encoders import OneHotEncoderForest
from forestpy.ml.mlp import MLPRegressor, MLPTrainer
from forestpy.ml.evaluation import kfold_cv, bootstrap_metric
from forestpy.ml.metrics import regression_report, rmse
from forestpy.utils import set_seed, get_logger
from forestpy.viz.style import apply_forest_style
from forestpy.viz.diagnostics import plot_learning_curve, plot_predicted_vs_observed

set_seed(42)
apply_forest_style()
log = get_logger('sessao_07')

df = load_pef_vinhedo(synthetic_fallback=True, n_synthetic=500)
log.info(f'Dataset: {df.shape[0]} árvores')
df[['dap', 'h', 'idade', 'classe']].head()

## 3. Baseline Clássico: Curtis (apenas DAP)

Forma:

$$\ln(H) = \beta_0 + \frac{\beta_1}{DAP} \quad\Rightarrow\quad H = e^{\beta_0 + \beta_1/DAP}$$

In [ ]:
def fit_predict_curtis(X_train, y_train, X_test):
    """Ajusta Curtis usando apenas DAP (primeira coluna de X)."""
    def f(dap, b0, b1):
        return np.exp(b0 + b1 / dap)
    popt, _ = curve_fit(f, X_train[:, 0], y_train, p0=[3.2, -8.0], maxfev=5000)
    return None, f(X_test[:, 0], *popt)

X_dap = df[['dap']].values
y_h = df['h'].values

cv_curtis = kfold_cv(
    fit_predict_curtis, X_dap, y_h,
    n_splits=5, model_name='Curtis (DAP)',
)
print(cv_curtis.summary())

**Resultado esperado:** R² entre 0.5 e 0.6 — Curtis é um bom modelo, mas explica apenas metade da variância da altura, porque ignora idade e classe.

## 4. Engenharia de Features: One-Hot Encoding

A variável `classe` é categórica e nominal (não há ordem matemática entre I, II, III no sentido estrito do modelo). Convertemos para representação one-hot.

In [ ]:
encoder = OneHotEncoderForest()
classe_oh = encoder.fit_transform(df, cols=['classe'])

log.info(f'Classes aprendidas: {encoder.categories_["classe"]}')
log.info(f'Colunas geradas:   {encoder.column_names_}')

# Visualiza as primeiras linhas
pd.DataFrame(classe_oh[:5], columns=encoder.column_names_).assign(
    classe_original=df['classe'].head().values
)

In [ ]:
# Matriz de features completa para o MLP: [DAP, idade, classe_I, classe_II, classe_III]
X_full = np.column_stack([
    df['dap'].values,
    df['idade'].values,
    classe_oh,
]).astype(np.float32)

y_h_f32 = y_h.astype(np.float32)
log.info(f'Shape de X_full: {X_full.shape}')
log.info(f'Features: DAP, idade, classe_I, classe_II, classe_III')

## 5. MLP Multi-feature

Arquitetura: **5 → 64 → 32 → 1** com dropout 0.15. Mais larga que a Sessão 06 para acomodar a maior complexidade das interações.

In [ ]:
# Holdout para visualização da curva de aprendizado
rng = np.random.default_rng(42)
idx = rng.permutation(len(df))
tr_idx, te_idx = idx[:400], idx[400:]

sc_demo = StandardScalerForest()
X_tr_demo = sc_demo.fit_transform(X_full[tr_idx]).astype(np.float32)
X_te_demo = sc_demo.transform(X_full[te_idx]).astype(np.float32)
y_tr_demo = y_h_f32[tr_idx]
y_te_demo = y_h_f32[te_idx]

n_val = int(0.2 * len(X_tr_demo))
model_demo = MLPRegressor(input_dim=5, hidden_dims=[64, 32], dropout=0.15)
log.info(f'Parâmetros treináveis: {model_demo.count_parameters()}')

trainer_demo = MLPTrainer(model_demo, learning_rate=1e-3, weight_decay=1e-5)
history = trainer_demo.fit(
    X_tr_demo[:-n_val], y_tr_demo[:-n_val],
    X_tr_demo[-n_val:], y_tr_demo[-n_val:],
    epochs=250, batch_size=32, patience=25, verbose=False,
)
log.info(f'Treino: {len(history.train_loss)} épocas (melhor: {history.best_epoch})')

In [ ]:
fig_lc = plot_learning_curve(
    history.train_loss, history.val_loss,
    best_epoch=history.best_epoch,
    title='Curva de Aprendizado — MLP Hipsométrica',
)
fig_lc.savefig('../reports/figures/07_curva_aprendizado.png')
plt.show()

## 6. Comparação Visual e Estatística

In [ ]:
# 5-fold CV para o MLP (mesma rotina do baseline)
def fit_predict_mlp(X_train, y_train, X_test):
    sc = StandardScalerForest()
    Xtr = sc.fit_transform(X_train).astype(np.float32)
    Xte = sc.transform(X_test).astype(np.float32)
    ytr = y_train.astype(np.float32)

    n_val = int(0.2 * len(Xtr))
    model = MLPRegressor(input_dim=Xtr.shape[1], hidden_dims=[64, 32], dropout=0.15)
    tr = MLPTrainer(model, learning_rate=1e-3, weight_decay=1e-5)
    tr.fit(
        Xtr[:-n_val], ytr[:-n_val],
        Xtr[-n_val:], ytr[-n_val:],
        epochs=250, batch_size=32, patience=25, verbose=False,
    )
    return None, tr.predict(Xte)

log.info('Executando 5-fold CV no MLP...')
cv_mlp = kfold_cv(fit_predict_mlp, X_full, y_h_f32,
                  n_splits=5, model_name='MLP (DAP+idade+classe)')
print(cv_mlp.summary())

In [ ]:
# Tabela comparativa
comparativo = pd.DataFrame([
    {
        'Modelo': 'Curtis (só DAP)',
        'Features': 1,
        'RMSE médio': cv_curtis.mean_metrics['rmse'],
        'R² médio': cv_curtis.mean_metrics['r2'],
        'MAPE médio (%)': cv_curtis.mean_metrics['mape'],
    },
    {
        'Modelo': 'MLP (DAP+idade+classe)',
        'Features': 5,
        'RMSE médio': cv_mlp.mean_metrics['rmse'],
        'R² médio': cv_mlp.mean_metrics['r2'],
        'MAPE médio (%)': cv_mlp.mean_metrics['mape'],
    },
]).round(4)

ganho_rmse = (1 - cv_mlp.mean_metrics['rmse'] / cv_curtis.mean_metrics['rmse']) * 100
log.info(f'\nGanho do MLP sobre Curtis: {ganho_rmse:.1f}% de redução do RMSE')

comparativo.to_csv('../reports/tables/07_curtis_vs_mlp.csv', index=False)
comparativo

In [ ]:
# IC bootstrap para o veredito estatístico
# Predições no holdout para bootstrap
y_pred_curtis = curtis(df['dap'].values)
y_pred_mlp_all = trainer_demo.predict(sc_demo.transform(X_full).astype(np.float32))

ic_curtis = bootstrap_metric(y_h, y_pred_curtis, rmse, n_bootstrap=2000)
ic_mlp = bootstrap_metric(y_h, y_pred_mlp_all, rmse, n_bootstrap=2000)

log.info('Bootstrap RMSE (n=2000):')
log.info(f'  Curtis: {ic_curtis["mean"]:.4f} '
         f'(IC 95%: [{ic_curtis["ci_lower"]:.4f}, {ic_curtis["ci_upper"]:.4f}])')
log.info(f'  MLP:    {ic_mlp["mean"]:.4f} '
         f'(IC 95%: [{ic_mlp["ci_lower"]:.4f}, {ic_mlp["ci_upper"]:.4f}])')

if ic_mlp['ci_upper'] < ic_curtis['ci_lower']:
    log.info('\n✅ MLP supera Curtis com significância estatística (IC não sobrepostos)')
elif ic_curtis['ci_upper'] < ic_mlp['ci_lower']:
    log.info('\n❌ Curtis supera MLP')
else:
    log.info('\n➖ Sem diferença significativa')

In [ ]:
# Visualização: comparação de RMSE
fig_comp, axes = plt.subplots(1, 2, figsize=(14, 5))

modelos = ['Curtis (DAP)', 'MLP (DAP+idade+classe)']
rmses = [cv_curtis.mean_metrics['rmse'], cv_mlp.mean_metrics['rmse']]
stds = [cv_curtis.std_metrics['rmse'], cv_mlp.std_metrics['rmse']]
axes[0].bar(modelos, rmses, yerr=stds, capsize=10, color=['#a04a2c', '#2d5016'])
axes[0].set_ylabel('RMSE (m)')
axes[0].set_title('RMSE em 5-fold CV', fontweight='bold')

# Bootstrap IC
axes[1].errorbar(
    modelos,
    [ic_curtis['mean'], ic_mlp['mean']],
    yerr=[
        [ic_curtis['mean']-ic_curtis['ci_lower'], ic_mlp['mean']-ic_mlp['ci_lower']],
        [ic_curtis['ci_upper']-ic_curtis['mean'], ic_mlp['ci_upper']-ic_mlp['mean']],
    ],
    fmt='o', markersize=12, capsize=10, capthick=2, linewidth=2,
    color='#2d5016',
)
axes[1].set_ylabel('RMSE (m)')
axes[1].set_title('Bootstrap IC 95% (n=2000)', fontweight='bold')

fig_comp.tight_layout()
fig_comp.savefig('../reports/figures/07_comparativo_estatistico.png')
plt.show()

In [ ]:
# Predito vs Observado dos dois modelos lado a lado
fig_po, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, nome in [
    (axes[0], y_pred_curtis, 'Curtis (só DAP)'),
    (axes[1], y_pred_mlp_all, 'MLP (multi-feature)'),
]:
    ax.scatter(y_h, y_pred, alpha=0.4, edgecolor='white', linewidth=0.3)
    lims = [min(y_h.min(), y_pred.min()), max(y_h.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlabel('Altura observada (m)')
    ax.set_ylabel('Altura predita (m)')
    ax.set_title(nome, fontweight='bold')
    ax.set_aspect('equal')

fig_po.suptitle('Predito vs. Observado — Hipsometria', fontsize=14, fontweight='bold')
fig_po.tight_layout()
fig_po.savefig('../reports/figures/07_predito_vs_observado.png')
plt.show()

## 7. Análise por Classe de Sítio

A vantagem do MLP fica especialmente clara quando estratificamos os erros por classe de sítio — Curtis é forçado a usar a mesma curva para todos, enquanto o MLP adapta-se a cada estrato.

In [ ]:
# Calcula RMSE por classe para cada modelo
erros_por_classe = []
for classe in ['I', 'II', 'III']:
    mask = df['classe'] == classe
    rmse_curtis_c = np.sqrt(np.mean((y_h[mask] - y_pred_curtis[mask])**2))
    rmse_mlp_c = np.sqrt(np.mean((y_h[mask] - y_pred_mlp_all[mask])**2))
    erros_por_classe.append({
        'Classe': classe,
        'n árvores': mask.sum(),
        'RMSE Curtis': rmse_curtis_c,
        'RMSE MLP': rmse_mlp_c,
        'Ganho (%)': (1 - rmse_mlp_c / rmse_curtis_c) * 100,
    })

pd.DataFrame(erros_por_classe).round(3)

In [ ]:
# Visualização: resíduos por classe
fig_cls, axes = plt.subplots(1, 2, figsize=(14, 5))

df_residuos = pd.DataFrame({
    'classe': df['classe'].values,
    'resíduo_curtis': y_h - y_pred_curtis,
    'resíduo_mlp': y_h - y_pred_mlp_all,
})

for ax, col, titulo in [
    (axes[0], 'resíduo_curtis', 'Curtis (só DAP)'),
    (axes[1], 'resíduo_mlp', 'MLP (multi-feature)'),
]:
    df_residuos.boxplot(column=col, by='classe', ax=ax, grid=True)
    ax.axhline(0, color='red', ls='--', lw=1)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Classe de Sítio')
    ax.set_ylabel('Resíduo (m)')

fig_cls.suptitle('')
fig_cls.tight_layout()
fig_cls.savefig('../reports/figures/07_residuos_por_classe.png')
plt.show()

**🔍 Interpretação:** os boxplots de resíduos do Curtis mostram **bias sistemático por classe** (medianas deslocadas do zero — superestima sítios ruins, subestima sítios bons), enquanto o MLP centra os resíduos em zero para todas as classes. Esse é o tipo de viés que Redes Neurais corrigem naturalmente ao incorporar a variável de estratificação como feature.

## 8. Síntese: Quando ML Vale a Pena?

### Comparativo das duas sessões

| Aspecto | Sessão 06 (Volumetria) | Sessão 07 (Hipsometria) |
|---|---|---|
| Baseline | Schumacher-Hall (forma correta) | Curtis (sub-especificado) |
| Features do baseline | DAP + H | DAP |
| Features do MLP | DAP + H (iguais) | DAP + idade + classe |
| Resultado | MLP perde | MLP vence claramente |

### Lição metodológica

Comparando as duas sessões fica explícito o **princípio fundamental**: Redes Neurais não são *intrinsecamente* superiores a modelos clássicos. Elas oferecem vantagem quando há **mais informação disponível do que a forma funcional clássica consegue absorver**.

Em campo, o engenheiro florestal raramente dispõe de equações multi-variáveis ajustadas para cada combinação de classe e idade. O MLP integra essa estratificação de forma automática, capturando interações que exigiriam dezenas de equações segmentadas.

### Próxima sessão (08): Classificação de Sítio via Deep Learning

Saímos da regressão e entramos na **classificação multiclasse** — predizer a classe de sítio (I, II, III) a partir das variáveis dendrométricas. Tarefa onde modelos paramétricos clássicos não competem diretamente, e o MLP encontra seu espaço natural.